# Task A.5 — Out-of-Sample Prediction Performance (Gradient Boosting)

Reports the required pooled RMSE and mean monthly Spearman rank correlation for Gradient Boosting in validation and test, plus the pooled OLS comparison on the test sample.

**Richer model:** Gradient Boosting Regressor. The assignment requires exactly one richer model to be compared with pooled OLS. The richer model is tuned on validation data only and then retained at its training-sample fit.

## Documented analysis plan

| Component | Choice |
|---|---|
| Target | Next-month stock excess return from `ret_excess_t` |
| Benchmark | Pooled OLS, fitted on training + validation |
| Richer model | Gradient Boosting Regressor |
| Tuning criterion | Validation RMSE (equivalent to validation MSE ordering) |
| Hyperparameters | `n_estimators`, `learning_rate`, `max_depth`, `min_samples_leaf` |
| Missing data | Median imputation fitted on training data |
| Predictors | Same 18 numeric characteristics for both models |
| Train | Jan 1990–Dec 2014 |
| Validation | Jan 2015–Dec 2018 |
| Test | Jan 2019–Nov 2022 |


In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
import statsmodels.api as sm

DATA_DIR = Path(".")
PANEL_FILE = DATA_DIR / "FINANCE384_assignmentA_development_panel.csv"
MARKET_FILE = DATA_DIR / "FINANCE384_market.csv"
DICT_FILE = DATA_DIR / "FINANCE384_stock_month_data_dictionary.csv"

panel = pd.read_csv(PANEL_FILE, parse_dates=["date"])
market = pd.read_csv(MARKET_FILE, parse_dates=["date"])
data_dictionary = pd.read_csv(DICT_FILE)

# Create next-month target without carrying a return across a missing stock-month.
panel = panel.sort_values(["permno", "date"]).reset_index(drop=True)
panel["target_next"] = panel.groupby("permno")["ret_excess_t"].shift(-1)

all_dates = pd.Series(sorted(panel["date"].dropna().unique()))
next_date_map = {all_dates.iloc[i]: all_dates.iloc[i+1] for i in range(len(all_dates)-1)}
panel["expected_next_date"] = panel["date"].map(next_date_map)
panel["actual_next_date"] = panel.groupby("permno")["date"].shift(-1)
panel.loc[panel["actual_next_date"] != panel["expected_next_date"], "target_next"] = np.nan

# Same base predictor information for OLS and Gradient Boosting.
FEATURES = [
    "size", "bm", "mom12_2", "vol12", "beta60", "ivol60", "turnover",
    "dollar_volume", "amihud_illiq", "divyield", "gross_profit", "roe",
    "asset_growth", "leverage", "accruals", "mkt_12m", "mkt_vol_12m",
    "down_market"
]

TRAIN_START, TRAIN_END = "1990-01-31", "2014-12-31"
VAL_START, VAL_END = "2015-01-31", "2018-12-31"
TEST_START, TEST_END = "2019-01-31", "2022-11-30"

def get_period(start, end):
    return panel.loc[
        panel["date"].between(start, end) & panel["target_next"].notna()
    ].copy()

train = get_period(TRAIN_START, TRAIN_END)
val = get_period(VAL_START, VAL_END)
test = get_period(TEST_START, TEST_END)

# Missing-data rule: median imputation fitted on training data only.
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(train[FEATURES])
X_val = imputer.transform(val[FEATURES])
X_test = imputer.transform(test[FEATURES])

y_train = train["target_next"].to_numpy()
y_val = val["target_next"].to_numpy()
y_test = test["target_next"].to_numpy()

# Gradient Boosting tuning grid. Validation MSE/RMSE is the sole selection criterion.
GB_GRID = [
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 100},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 100},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 100},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 100},
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 3, "min_samples_leaf": 100},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 3, "min_samples_leaf": 100},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 100},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 100},
]

gb_tuning = []
for params in GB_GRID:
    model = GradientBoostingRegressor(
        **params, loss="squared_error", random_state=384
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    gb_tuning.append({
        **params,
        "validation_MSE": np.mean((y_val - pred)**2),
        "validation_RMSE": np.sqrt(np.mean((y_val - pred)**2))
    })

gb_tuning = pd.DataFrame(gb_tuning)
best_idx = gb_tuning["validation_RMSE"].idxmin()
best_params = gb_tuning.loc[best_idx, GB_GRID[0].keys()].to_dict()

# Refit the selected specification on TRAIN ONLY, as required by the protocol.
gb_model = GradientBoostingRegressor(
    **{k: (int(v) if k in ["n_estimators", "max_depth", "min_samples_leaf"] else float(v))
       for k, v in best_params.items()},
    loss="squared_error",
    random_state=384
)
gb_model.fit(X_train, y_train)

# Pooled OLS benchmark on combined train + validation.
ols_imputer = SimpleImputer(strategy="median")
X_ols_trainval = ols_imputer.fit_transform(
    pd.concat([train[FEATURES], val[FEATURES]])
)
y_ols_trainval = pd.concat([train["target_next"], val["target_next"]]).to_numpy()
X_ols_test = ols_imputer.transform(test[FEATURES])

ols_model = sm.OLS(
    y_ols_trainval,
    sm.add_constant(X_ols_trainval, has_constant="add")
).fit()

def ols_predict(d):
    return ols_model.predict(
        sm.add_constant(ols_imputer.transform(d[FEATURES]), has_constant="add")
    )

def gb_predict(d):
    return gb_model.predict(imputer.transform(d[FEATURES]))

def prediction_metrics(pred, d):
    errors = d["target_next"].to_numpy() - pred
    rhos = []
    tmp = d[["date", "target_next"]].copy()
    tmp["pred"] = pred
    for _, g in tmp.groupby("date"):
        if len(g) >= 2:
            rhos.append(spearmanr(g["pred"], g["target_next"]).statistic)
    return {
        "RMSE": float(np.sqrt(np.mean(errors**2))),
        "Mean monthly Spearman": float(np.nanmean(rhos)),
        "Months": int(d["date"].nunique()),
        "Stock-month rows": int(len(d))
    }

def make_quintile_portfolios(pred, d):
    z = d[["date", "target_next"]].copy()
    z["prediction"] = pred
    out = []
    for date, g in z.groupby("date"):
        g = g.sort_values("prediction").reset_index(drop=True)
        n = len(g)
        quintile = 1 + np.floor(5 * np.arange(n) / n).astype(int)
        qret = g.groupby(quintile)["target_next"].mean()
        out.append({
            "formation_date": date,
            "P1": qret.get(1, np.nan),
            "P5": qret.get(5, np.nan),
            "P5-P1": qret.get(5, np.nan) - qret.get(1, np.nan),
            "N": n
        })
    return pd.DataFrame(out).sort_values("formation_date").reset_index(drop=True)

def market_alpha(portfolio):
    m = portfolio.copy()
    m["realized_date"] = m["formation_date"].map(next_date_map)
    m = m.merge(market.rename(columns={"date": "realized_date"}),
                on="realized_date", how="inner")
    reg = sm.OLS(m["P5-P1"], sm.add_constant(m["mktrf"], has_constant="add")).fit()
    spread = m["P5-P1"].dropna()
    return {
        "Monthly P5-P1": float(spread.mean()),
        "P5-P1 t-stat": float(spread.mean() / (spread.std(ddof=1) / np.sqrt(len(spread)))),
        "Market alpha": float(reg.params["const"]),
        "Alpha t-stat": float(reg.tvalues["const"]),
        "Months": int(len(spread))
    }


In [ ]:
print('Train:', train.shape)
print('Validation:', val.shape)
print('Test:', test.shape)
print('\nPredictors:')
print(FEATURES)

## Gradient Boosting hyperparameter tuning

In [ ]:
display(gb_tuning.sort_values('validation_RMSE').reset_index(drop=True))
print('Selected parameters:', best_params)

## Required prediction metrics

In [ ]:

gb_val = prediction_metrics(gb_predict(val), val)
gb_test = prediction_metrics(gb_predict(test), test)
ols_test = prediction_metrics(ols_predict(test), test)

metrics_table = pd.DataFrame([
    {"Model / period": "Gradient Boosting — validation", **gb_val},
    {"Model / period": "Gradient Boosting — test", **gb_test},
    {"Model / period": "Pooled OLS — test", **ols_test},
])
metrics_table


The test comparison uses the same valid stock-month rows. Report both the number of months and stock-month observations alongside each metric.